In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import multiprocessing
import os
import pickle

try:
    cores = int(os.environ["SLURM_JOB_CPUS_PER_NODE"])
except:
    cores = multiprocessing.cpu_count()

os.environ["XLA_FLAGS"] = "--xla_force_host_platform_device_count={}".format(cores)

import jax
import jax.numpy as jnp
import numpy as np
from jax import jit
from jaxoplanet import orbits
from jaxoplanet.light_curves import LimbDarkLightCurve
from jaxopt import ScipyMinimize
from tensorflow_probability.substrates.jax.bijectors import Log
from tensorflow_probability.substrates.jax.distributions import (
    Normal,
)

from gallifrey.inference import log_likelihood_function
from gallifrey.kernelsearch import kernel_summary, get_trainables
from gallifrey.mcmc import nuts_warmup, run_mcmc
from gallifrey.util import example_lightcurve, example_transit_model

rng_key = jax.random.PRNGKey(42)

## LOAD DATA AND MODEL

In [3]:
example = example_lightcurve()

model_name = example["name"]
with open(f"../data/processed/toy_data/gp_models/{model_name}", "rb") as file:
    model = pickle.load(file)

summary = kernel_summary(model, to_latex=False)

Kernel Summary

Kernel Structure: Periodic • Linear • Linear + Periodic
with obs_stddev = 1.00000e-03 (Trainable : False)

Kernel               Property             Value                Trainable 
--------------------------------------------------------------------------------
Periodic            lengthscale          1.57304e+02          True      

                    variance             1.04436e-04          True      

                    period               1.81344e+04          True      
--------------------------------------------------------------------------------
Linear              variance             1.00000e+00          False     
--------------------------------------------------------------------------------
Linear              variance             1.00000e+00          False     
--------------------------------------------------------------------------------
Periodic            lengthscale          2.15577e-01          True      

                    variance          

## CREATE TRANSIT MODEL

In [4]:
@jit
def lc_model(t, params):
    orbit = orbits.keplerian.Body(
        period=15,
        radius=params[0],
        inclination=jnp.deg2rad(89),
        time_transit=0,
    )

    lc = LimbDarkLightCurve([params[1], params[2]]).light_curve(orbit, t=t)
    return lc

## FIT LC

In [5]:
initial_position = {
    "gp_parameter": get_trainables(
        model, unconstrain=True
    ),  # gp is fitted in unconstrained space, so set initial value to that
    "lc_parameter": jnp.asarray([0.2, 0.2, 0.5]),
}

param_priors = {
    "gp_parameter": Normal(loc=initial_position["gp_parameter"], scale=1),
    "lc_parameter": Normal(
        loc=initial_position["lc_parameter"],
        scale=[0.1, 0.3, 0.2],
    ),
}

In [6]:
log_likelihood = log_likelihood_function(
    model,
    lc_model,
    example["t_sample"],
    example["lc_sample"],
    example["transit_mask_sample"],
    fix_gp=False,
    compile=True,
)


@jit
def log_priors(params):
    gp_log_priors = param_priors["gp_parameter"].log_prob(params["gp_parameter"])
    lc_log_priors = param_priors["lc_parameter"].log_prob(params["lc_parameter"])
    return jnp.sum(gp_log_priors) + jnp.sum(lc_log_priors)


@jit
def log_probability(params):
    return log_likelihood(params) + log_priors(params)


neg_log_probability = jit(lambda params: -log_probability(params))

In [7]:
lbfgsb = ScipyMinimize(fun=neg_log_probability, method="l-bfgs-b")
lbfgsb_sol = lbfgsb.run(initial_position)
print("Best fit parameter: ", lbfgsb_sol.params["lc_parameter"])

Best fit parameter:  [0.10039986 0.10397246 0.48633141]


## RUN MCMC

In [ ]:
# Adapted from BlackJax's introduction notebook.
num_adapt = 500
num_samples = 500
num_chains = cores

rng_key, warmup_key, sample_key = jax.random.split(rng_key, 3)

state, parameters = nuts_warmup(
    warmup_key,
    log_probability,
    initial_position,
    num_steps=num_adapt,
)

initial_positions = {
    "gp_parameter": jnp.tile(state.position["gp_parameter"], (num_chains, 1)),
    "lc_parameter": jnp.tile(state.position["lc_parameter"], (num_chains, 1)),
}

final_state, state_history, info_history = run_mcmc(
    sample_key,
    log_probability,
    parameters,
    initial_positions,
    num_steps=num_samples,
)

In [ ]:
np.save(
    f"../data/processed/toy_data/mcmc_chains/{model_name}_parameter.npy",
    np.array(
        state_history.position["lc_parameter"].reshape(
            -1, state_history.position["lc_parameter"].shape[-1]
        )
    ),
)
np.save(
    f"../data/processed/toy_data/mcmc_chains/{model_name}_parameter_gp.npy",
    np.array(
        state_history.position["gp_parameter"].reshape(
            -1, state_history.position["gp_parameter"].shape[-1]
        )
    ),
)